In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown,display

In [ ]:
load_dotenv(override=True)


In [ ]:
google_api_key = os.getenv('GEMINI_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:2]}")
else:
    print("OpenRouter API Key not set")
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set")



In [ ]:
request="Please come up with the challenging question that I can ask a number of LLMS to evaluate their moral values spectrem where they stand morally and philosophically"
request += "Answer only with the question, no explanation."
message=[{'role':'user','content':request}]

In [ ]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
openrouter=OpenAI(base_url=OPENROUTER_BASE_URL,api_key=openrouter_api_key)
response=openrouter.chat.completions.create(
    model="openai/gpt-oss-20b:free",
    messages=message
)
question=response.choices[0].message.content
print(question)

In [ ]:
competitiors=[]
answers=[]
messages=[{"role":"user","content":question}]

In [ ]:
model_name="openai/gpt-oss-20b:free"

response= openrouter.chat.completions.create(model=model_name,messages=messages)
answer=response.choices[0].message.content

display(Markdown(answer))
competitiors.append(model_name)
answers.append(answer)

In [ ]:
model_name = "gemini-2.5-flash"
gemini=OpenAI(api_key=google_api_key,base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
response=gemini.chat.completions.create(model=model_name,messages=messages)
answer=response.choices[0].message.content
display(Markdown(answer))
competitiors.append(model_name)
answers.append(answer)

In [ ]:
!ollama pull llama3.2

In [ ]:
ollama=OpenAI(base_url='http://localhost:11434/v1',api_key='ollama')
model_name="llama3.2"
response=ollama.chat.completions.create(model=model_name,messages=messages)
answer=response.choices[0].message.content

display(Markdown(answer))
competitiors.append(model_name)
answers.append(answer)

In [ ]:
print(competitiors)
print(answers)

In [ ]:
for competitior,answer in zip(competitiors,answers):
    print(f"Competitior:{competitior}\n\n{answer}")


In [ ]:
together=""
for index,answer in enumerate(answers):
    together+=f"#Response from competitor {index}\n\n"
    together+=answer+"\n\n"

In [ ]:
print(together)

In [ ]:
judge = f"""You are judging a competition between {len(competitiors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [ ]:
print(judge)

In [ ]:
judge_messages=[{'role':'user','content':judge}]

In [ ]:
open_router1=OpenAI(base_url=OPENROUTER_BASE_URL,api_key=openrouter_api_key)

In [ ]:
response=open_router1.chat.completions.create(
    model="openai/gpt-oss-120b:free",
    messages=judge_messages,
)
results=response.choices[0].message.content
print(results)

In [ ]:
result_dict=json.loads(results)
ranks=result_dict["results"]
for index,result in enumerate(ranks):
    competitior=competitiors[int(result)-1]
    print(f"Rank{index+1}:{competitior}")